# 03. Product-Level Reorder Frequency & Restock Prediction

**Lead Responsible**: Peiris W.S.V (`IT24103792`)  
**Module**: IT3091 – Machine Learning | Group 2026-AI-15  
**Track**: Industry Explorer Track  

---

## Objectives
1. Engineer product-level features from transaction histories (reorder count, repurchase lag from `days_since_prior_order`, department velocities).
2. Train baseline **Logistic Regression** classifier to predict high-frequency reorder items.
3. Benchmark non-linear classification models: **Decision Tree** and **Random Forest**.
4. Evaluate performance using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

In [2]:
import sys
import time
from pathlib import Path

# Resolve project root dynamically
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn Modeling, Tuning & Validation Modules
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_val_score, 
    RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    HistGradientBoostingClassifier
)

# Project modular utilities from src/
from src.utils import setup_plot_style, save_figure, export_summary_table

setup_plot_style()
print(f"[OK] Python Interpreter : {sys.executable}")
print(f"[OK] Project Root       : {PROJECT_ROOT}")


Matplotlib is building the font cache; this may take a moment.


[OK] Python Interpreter : /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics/.venv/bin/python3.13
[OK] Project Root       : /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics


---
## 1. Data Ingestion: Train and Test Partitions
We ingest `train_transactions.parquet` and `test_transactions.parquet` generated by the preprocessing pipeline.


In [3]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = DATA_DIR / "train_transactions.parquet"
TEST_PATH = DATA_DIR / "test_transactions.parquet"

assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}. Check data/processed/"
assert TEST_PATH.exists(), f"Missing: {TEST_PATH}. Check data/processed/"

print(f"Loading partitions from {DATA_DIR}...")
t0 = time.time()
df_train = pd.read_parquet(TRAIN_PATH)
df_test = pd.read_parquet(TEST_PATH)

print(f"✅ Loaded Train Partition : {len(df_train):,} rows ({time.time() - t0:.2f}s)")
print(f"✅ Loaded Test Partition  : {len(df_test):,} rows")


Loading partitions from /Users/supunvidarshana/Desktop/My projects/supermarket-basket-reorder-analytics/data/processed...
✅ Loaded Train Partition : 1,615,600 rows (3.65s)
✅ Loaded Test Partition  : 403,901 rows
